In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import xarray as xr
from shapely.geometry import Polygon
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import pickle
import h5py

from config import LOCATIONS, OUTPUT_FILE_XARRAY_INIT, DEG_RES

In [ ]:
FILE_SHAPE = r"input_data\ne_110m_admin_0_countries\ne_110m_admin_0_countries.shp"
WORLD = gpd.read_file(FILE_SHAPE)[['geometry', 'NAME_EN', "ISO_A2", "ISO_A3", "CONTINENT"]]

In [ ]:
# --- 1. Define special-case countries and their codes ---
special_cases = {
    'United Kingdom': {'ISO_A2': 'GB', 'ISO_A3': 'GBR'},
    'World': {'ISO_A2': 'WORLD', 'ISO_A3': 'WLD'},
    'Europe': {'ISO_A2': 'EU', 'ISO_A3': 'EUR'},
    'Norway': {'ISO_A2': 'NO', 'ISO_A3': 'NOR'},
    'France': {'ISO_A2': 'FR', 'ISO_A3': 'FRA'},
    'Kosovo': {'ISO_A2': 'XK', 'ISO_A3': 'XKX'},
    'Somaliland': {'ISO_A2': 'SO', 'ISO_A3': 'SOM'}
}

# --- 2. Work on a safe copy of WORLD ---
WORLD = WORLD.copy()

# --- 3. Clean invalid codes ---
WORLD['ISO_A2'] = WORLD['ISO_A2'].replace('-99', np.nan)
WORLD['ISO_A3'] = WORLD['ISO_A3'].replace('-99', np.nan)

# --- 4. Fill special cases (vectorized, no apply) ---
for name, codes in special_cases.items():
    mask = WORLD['NAME_EN'] == name
    WORLD.loc[mask, 'ISO_A2'] = codes['ISO_A2']
    WORLD.loc[mask, 'ISO_A3'] = codes['ISO_A3']

# --- 5. Check results ---
missing = WORLD[WORLD['ISO_A2'].isna() | WORLD['ISO_A3'].isna()]
print(f"Remaining missing ISO codes: {len(missing)}")
if not missing.empty:
    print(missing[['NAME_EN', 'ISO_A2', 'ISO_A3']])

In [ ]:
WORLD.loc[WORLD['NAME_EN'].str.contains('Norway', case=False, na=False)]

## 1. Import data from Mingolla et al, in terms of synthetic nitrogen demand, credits to:
Mingolla, S., & Rosa, L. (2025). Low-carbon ammonia production is essential for resilient and sustainable agriculture. Nature Food, 1-12.

**Note for PtX Project**: While the original study focused on ammonia/nitrogen demand, for this PtX (Power-to-X) adaptation, we include synthetic nitrogen data for reference and visualization purposes only. The optimization will run on **ALL pixels globally**, not limited by nitrogen demand, to assess optimal PtX production locations for various products (methanol, DAC, etc.).
https://www.nature.com/articles/s43016-025-01125-y

In [ ]:
# Load data
with h5py.File('input_data/input_data_Mingolla/2020_synthetic_nitrogen_tonnes.h5', 'r') as hf:
    synthetic_nitrogen = hf['dataset_name'][:]

# Dataset Overview
print("Synthetic Nitrogen Dataset Summary:")
print(f"Shape: {synthetic_nitrogen.shape} (rows x cols)")
print(f"Data type: {synthetic_nitrogen.dtype}")

# Total and Mean Demand (tons per pixel)
total_nitrogen = np.sum(synthetic_nitrogen)
mean_nitrogen = np.mean(synthetic_nitrogen)
print(f"\nTotal Global Synthetic Nitrogen Demand: {total_nitrogen:,.2f} tons")
print(f"Mean Demand per Pixel: {mean_nitrogen:.2f} tons")

# Area Correction: Convert to tons/km2
deg_to_rad = np.pi / 180
delta_deg = 1 / 12
earth_radius_km = 6371.0
delta_rad = delta_deg * deg_to_rad

latitudes = np.linspace(90 - delta_deg / 2, -90 + delta_deg / 2, synthetic_nitrogen.shape[0])
pixel_area = (earth_radius_km ** 2) * delta_rad * delta_rad * np.cos(latitudes * deg_to_rad)
pixel_area = pixel_area[:, np.newaxis]

nitrogen_per_km2 = synthetic_nitrogen / pixel_area

# Positive Pixel Statistics (tons/km2)
positive_values = nitrogen_per_km2[nitrogen_per_km2 > 0]

print("\nSynthetic Nitrogen Intensity (tons/km2) — Positive Pixels Only:")
print(f"Min: {np.min(positive_values):.2f}")
print(f"5th Percentile: {np.percentile(positive_values, 5):.2f}")
print(f"Median: {np.median(positive_values):.2f}")
print(f"Mean: {np.mean(positive_values):.2f}")
print(f"95th Percentile: {np.percentile(positive_values, 95):.2f}")
print(f"Max: {np.max(positive_values):,.2f}")

# Pixel Coverage Summary
total_pixels = synthetic_nitrogen.size
positive_pixels = positive_values.size
positive_ratio = positive_pixels / total_pixels

print("\nPixel Coverage:")
print(f"Positive pixels: {positive_pixels:,} of {total_pixels:,} ({positive_ratio:.2%})")

#### 1.1 Plot this data with the case studies considered

In [ ]:
# Plotting Map of Nitrogen Demand per km2
synthetic_nitrogen_nonzero = np.ma.masked_equal(nitrogen_per_km2, 0)

width_mm = 180
height_mm = 90
dpi = 300
width_inch = width_mm / 25.4
height_inch = height_mm / 25.4

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 6

fig = plt.figure(figsize=(width_inch, height_inch), dpi=dpi)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, linewidth=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linestyle='-', linewidth=0.25)
ax.set_extent([-150, 167, -56, 71], crs=ccrs.PlateCarree())

lons = np.linspace(-180, 180, synthetic_nitrogen_nonzero.shape[1])
lats = np.linspace(90, -90, synthetic_nitrogen_nonzero.shape[0])
#lon2d, lat2d = np.meshgrid(lons, lats)

heatmap = ax.pcolormesh(lons, lats, synthetic_nitrogen_nonzero,
                         cmap='viridis', vmax=8, transform=ccrs.PlateCarree())

# --- Add location points ---
for name, iso2, lat, lon in LOCATIONS:
    ax.scatter(lon, lat, s=16, c='white', edgecolor='red', linewidth=0.9,
               transform=ccrs.PlateCarree(), zorder=5)
    ax.text( lon + 1.3, lat, name.split("(", 1)[0].strip(), fontsize=5, fontweight='bold', 
            color='black', transform=ccrs.PlateCarree(), zorder=6,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.9, boxstyle='round,pad=0.2'))

cbar = fig.colorbar(
    heatmap, ax=ax, orientation='vertical',
    pad=0.02, aspect=30, shrink=0.4  #
)
cbar.set_label('Demand (t N km$^{-2}$ year $^{-1}$)', fontsize=7.5)
cbar.ax.tick_params(labelsize=6)  # adjust tick font

plt.tight_layout()
plt.savefig("figs/nitrogen_map_with_case_studies.png", dpi=300, bbox_inches='tight')
plt.show()

# 2. Store this data efficiently in an xarray, add also colum  for other data aspects

In [ ]:
# --- 1. Define original fine grid ---
lat_res = 1/12
lon_res = 1/12
nlat, nlon = synthetic_nitrogen.shape

lats = np.linspace(90 - lat_res/2, -90 + lat_res/2, nlat)
lons = np.linspace(-180 + lon_res/2, 180 - lon_res/2, nlon)

# --- 2. Crop to smaller region BEFORE coarsening ---
min_lat, max_lat = -58, 70
min_lon, max_lon = -140, 170

lat_mask = (lats <= max_lat) & (lats >= min_lat)
lon_mask = (lons >= min_lon) & (lons <= max_lon)

synthetic_nitrogen_cropped = synthetic_nitrogen[np.ix_(lat_mask, lon_mask)]
lats_cropped = lats[lat_mask]
lons_cropped = lons[lon_mask]

# --- 3. Create DataArray ---
da_nitrogen = xr.DataArray(
    synthetic_nitrogen_cropped,
    coords=[("lat", lats_cropped), ("lon", lons_cropped)],
    name="synthetic_nitrogen_tonnes",
    attrs={
        "units": "tonnes",
        "description": "Synthetic nitrogen fertilizer demand in 2020",
        "resolution": f"{lat_res}° x {lon_res}°"
    }
)

# --- 4. Coarsen to 2° ---
da_nitrogen_2deg = da_nitrogen.coarsen(
    lat=int(DEG_RES / lat_res),
    lon=int(DEG_RES / lon_res),
    boundary="trim"
).sum()

# --- 5. Assign new coordinates for 2° cells ---
lat_2deg = np.arange(max_lat - DEG_RES / 2, min_lat, -DEG_RES)
lon_2deg = np.arange(min_lon + DEG_RES / 2, max_lon, DEG_RES)

da_nitrogen_2deg = da_nitrogen_2deg.assign_coords(lat=lat_2deg, lon=lon_2deg)

# --- 6. Initialize Dataset ---
ds = xr.Dataset(coords={"lat": da_nitrogen_2deg.lat, "lon": da_nitrogen_2deg.lon})
ds["synthetic_nitrogen_tonnes"] = da_nitrogen_2deg
ds["synthetic_nitrogen_tonnes"].attrs = da_nitrogen.attrs
ds

# 3. Add additional info, like country, ISO2, land code etc.:

In [ ]:
# PtX-only grid initialization (no nitrogen data dependency)
min_lat, max_lat = -58, 70
min_lon, max_lon = -140, 170

lat_2deg = np.arange(max_lat - DEG_RES / 2, min_lat, -DEG_RES)
lon_2deg = np.arange(min_lon + DEG_RES / 2, max_lon, DEG_RES)

ds = xr.Dataset(coords={"lat": lat_2deg, "lon": lon_2deg})
ds


In [ ]:
# --- 1. Create centroid points for each grid cell ---
lons, lats = np.meshgrid(ds.lon.values, ds.lat.values)
lons_flat = lons.ravel()
lats_flat = lats.ravel()
centroids = [Point(lon, lat) for lon, lat in zip(lons_flat, lats_flat)]

# --- 2. Create temporary GeoDataFrame of centroids ---
gdf_centroids = gpd.GeoDataFrame(
    {"lat": lats_flat, "lon": lons_flat},
    geometry=centroids,
    crs="EPSG:4326"
)

# --- 3. Assign country info via spatial join ---
countries = WORLD[['geometry', 'NAME_EN', 'ISO_A2', 'CONTINENT']].copy()
gdf_joined = gpd.sjoin(gdf_centroids, countries, how="left", predicate='within')

# Identify rows where ISO code is missing after the spatial join
unmatched = gdf_joined[gdf_joined['ISO_A2'] == '-99']
print(unmatched['NAME_EN'].unique())

In [ ]:
gdf_joined[gdf_joined['NAME_EN']=='Netherlands']

In [ ]:
gdf_joined[gdf_joined['NAME_EN']=='Norway']

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# Raster input files
# ══════════════════════════════════════════════════════════════════════════════
FILE_LAND_CLASS  = r"input_data\land_cover\PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_EPSG-4326.tif"
FILE_PROTECTED   = r"input_data\land_cover\protected_areas.tif"
FILE_ELEVATION   = r"input_data\land_cover\ELE.tif"
FILE_SLOPE       = r"input_data\land_cover\slope_1KMmd_GMTEDmd.tif"
FILE_WIND_ON     = r"input_data\land_cover\gwa3_250_capacityfactor_IEC1.tif"
FILE_PV          = r"input_data\land_cover\PVOUT.tif"
FILE_ARIDITY     = r"input_data\land_cover\ai_et0.tif"
FILE_DIST_COAST  = r"input_data\land_cover\dist_coast.tif"
FILE_POP_DENSITY = r"input_data\land_cover\pop_density.tif"

# ══════════════════════════════════════════════════════════════════════════════
# Filter thresholds
# ══════════════════════════════════════════════════════════════════════════════
MIN_ELEVATION_M   = -50
MAX_ELEVATION_M   = 2000
MAX_SLOPE_DEG     = 15
MIN_CF_PV         = 0.17
MIN_CF_WIND_ON    = 0.25
MIN_ARIDITY_INDEX = 0.03    # actual P/PET; ai_et0.tif stores values × 10 000
MAX_DIST_COAST_KM = 300     # GMT convention: negative = inland, positive = ocean
MIN_POP_DENSITY   = 0.0001    # p/km²; below = genuinely uninhabited

# Spatial thinning
GRID_DEG        = 2
MIN_PER_COUNTRY = 1
MAX_PER_COUNTRY = 100

# ══════════════════════════════════════════════════════════════════════════════
# Toggle filters / thinning
# ══════════════════════════════════════════════════════════════════════════════
USE_CF_FILTER       = False   # disable renewable-resource threshold filter
USE_WATER_FILTER    = False   # disable aridity + coastal remoteness filter
USE_SPATIAL_THINNING = False  # disable grid/country thinning

# ══════════════════════════════════════════════════════════════════════════════
# Land classes to exclude
# ══════════════════════════════════════════════════════════════════════════════
LAND_EXCLUDE = {"forest", "na", "sea", "urban"}

# ══════════════════════════════════════════════════════════════════════════════
# PROBAV LC100 mappings
# ══════════════════════════════════════════════════════════════════════════════
dict_land_type = {
    0:     "No input data available",
    20:    "Shrubs",
    30:    "Herbaceous vegetation",
    40:    "Cultivated and managed vegetation/agriculture (cropland)",
    50:    "Urban / built up",
    60:    "Bare / sparse vegetation",
    70:    "Snow and Ice",
    80:    "Permanent water bodies",
    90:    "Herbaceous wetland",
    100:   "Moss and lichen",
    111:   "Closed forest, evergreen needle leaf",
    112:   "Closed forest, evergreen, broad leaf",
    113:   "Closed forest, deciduous needle leaf",
    114:   "Closed forest, deciduous broad leaf",
    115:   "Closed forest, mixed",
    116:   "Closed forest, unknown",
    121:   "Open forest, evergreen needle leaf",
    122:   "Open forest, evergreen broad leaf",
    123:   "Open forest, deciduous needle leaf",
    124:   "Open forest, deciduous broad leaf",
    125:   "Open forest, mixed",
    126:   "Open forest, unknown",
    200:   "Open sea",
    -56.0: "Open sea",
}

dict_land_type_class = {
    "No input data available":      "na",
    "Shrubs":                       "low vegetation",
    "Herbaceous vegetation":        "low vegetation",
    "Herbaceous wetland":           "na",
    "Moss and lichen":              "low vegetation",
    "Bare / sparse vegetation":     "open",
    "Cultivated and managed vegetation/agriculture (cropland)": "agri",
    "Urban / built up":             "urban",
    "Snow and Ice":                 "na",
    "Permanent water bodies":       "na",
    "Closed forest, evergreen needle leaf":  "forest",
    "Closed forest, evergreen, broad leaf":  "forest",
    "Closed forest, deciduous needle leaf":  "forest",
    "Closed forest, deciduous broad leaf":   "forest",
    "Closed forest, mixed":                  "forest",
    "Closed forest, unknown":                "forest",
    "Open forest, evergreen needle leaf":    "forest",
    "Open forest, evergreen broad leaf":     "forest",
    "Open forest, deciduous needle leaf":    "forest",
    "Open forest, deciduous broad leaf":     "forest",
    "Open forest, mixed":                    "forest",
    "Open forest, unknown":                  "forest",
    "Open sea":                              "sea",
}

# ══════════════════════════════════════════════════════════════════════════════
# Helper functions
# ══════════════════════════════════════════════════════════════════════════════
def get_land_class(code):
    name = dict_land_type.get(code)
    if name is None:
        return np.nan
    return dict_land_type_class.get(name, np.nan)


def sample_raster(gdf, filepath, col_name, extra_nodata=(-999, -9999)):
    """Sample a raster at point coordinates; NaN-fills if file is missing."""
    if not Path(filepath).is_file():
        print(f"  WARNING: {filepath} not found -- '{col_name}' filled with NaN.")
        gdf[col_name] = np.nan
        return gdf
    with rasterio.open(filepath) as src:
        coords = list(zip(gdf["lon"], gdf["lat"]))
        gdf[col_name] = [x[0] for x in src.sample(coords)]
        nodata = src.meta.get("nodata")
        if nodata is not None:
            gdf[col_name] = gdf[col_name].replace(nodata, np.nan)
        for nd in extra_nodata:
            gdf[col_name] = gdf[col_name].replace(nd, np.nan)
        gdf[col_name] = gdf[col_name].astype(float)
    return gdf


# ══════════════════════════════════════════════════════════════════════════════
# 1. SAMPLE ALL RASTERS
# ══════════════════════════════════════════════════════════════════════════════
print("Sampling rasters ...")
gdf_joined = sample_raster(gdf_joined, FILE_LAND_CLASS,  "land_code")
gdf_joined = sample_raster(gdf_joined, FILE_PROTECTED,   "protected")
gdf_joined = sample_raster(gdf_joined, FILE_ELEVATION,   "elevation_m")
gdf_joined = sample_raster(gdf_joined, FILE_SLOPE,       "slope_deg")
gdf_joined = sample_raster(gdf_joined, FILE_WIND_ON,     "cf_wind_on")
gdf_joined = sample_raster(gdf_joined, FILE_PV,          "cf_pv")
gdf_joined = sample_raster(gdf_joined, FILE_ARIDITY,     "aridity_index")
gdf_joined = sample_raster(gdf_joined, FILE_DIST_COAST,  "dist_coast_km")
gdf_joined = sample_raster(gdf_joined, FILE_POP_DENSITY, "pop_density")

# ai_et0.tif stores values as uint16 × 10 000; value 0 = ocean/no-data
gdf_joined["aridity_index"] = np.where(
    gdf_joined["aridity_index"] == 0, np.nan, gdf_joined["aridity_index"] / 10_000
)

# ══════════════════════════════════════════════════════════════════════════════
# 2. DERIVE LAND CLASS
# ══════════════════════════════════════════════════════════════════════════════
gdf_joined["land_type"]     = gdf_joined["land_code"].apply(lambda c: dict_land_type.get(c, np.nan))
gdf_joined["land_class"]    = gdf_joined["land_code"].apply(get_land_class)
gdf_joined["on_land"]       = ((gdf_joined["land_code"] > 0) & (gdf_joined["land_code"] != 200)).astype(int)
gdf_joined["elig_pv_urban"] = np.where(gdf_joined["land_class"] == "urban", 1, 0)
gdf_joined["cf_wind_on"]    = np.where(gdf_joined["on_land"] == 0, 0, gdf_joined["cf_wind_on"])
gdf_joined["cf_pv"]         = np.where(gdf_joined["on_land"] == 0, 0, gdf_joined["cf_pv"])

# ══════════════════════════════════════════════════════════════════════════════
# 3. PIXEL FILTER MASKS
# ══════════════════════════════════════════════════════════════════════════════
gdf_joined["filt_land"]      = (~gdf_joined["land_class"].isin(LAND_EXCLUDE)).fillna(False)
gdf_joined["filt_protected"] = gdf_joined["protected"].isna() | (gdf_joined["protected"] <= 0)
gdf_joined["filt_elevation"] = (
    gdf_joined["elevation_m"].isna()
    | ((gdf_joined["elevation_m"] >= MIN_ELEVATION_M) & (gdf_joined["elevation_m"] <= MAX_ELEVATION_M))
)
gdf_joined["filt_slope"]     = gdf_joined["slope_deg"].isna() | (gdf_joined["slope_deg"] <= MAX_SLOPE_DEG)

# Disabled: renewable resource threshold filter
if USE_CF_FILTER:
    gdf_joined["filt_cf"] = (
        (gdf_joined["cf_pv"] >= MIN_CF_PV)
        | (gdf_joined["cf_wind_on"] >= MIN_CF_WIND_ON)
    )
else:
    gdf_joined["filt_cf"] = True

# Disabled: water availability filter
if USE_WATER_FILTER:
    coast_ok   = gdf_joined["dist_coast_km"].isna() | (gdf_joined["dist_coast_km"] >= -MAX_DIST_COAST_KM)
    aridity_ok = gdf_joined["aridity_index"].isna()  | (gdf_joined["aridity_index"] >= MIN_ARIDITY_INDEX)
    gdf_joined["filt_water"] = coast_ok | aridity_ok
else:
    gdf_joined["filt_water"] = True

gdf_joined["filt_pop"] = (
    gdf_joined["pop_density"].isna()
    | (gdf_joined["pop_density"] >= MIN_POP_DENSITY)
)

if "NAME_EN" in gdf_joined.columns:
    gdf_joined["filt_country"] = ~(
        ((gdf_joined["NAME_EN"].isna()) | (gdf_joined["NAME_EN"] == ""))
        & (gdf_joined["on_land"] == 1)
    )
else:
    gdf_joined["filt_country"] = True

# ══════════════════════════════════════════════════════════════════════════════
# 4. COMBINED MASK
# ══════════════════════════════════════════════════════════════════════════════
gdf_joined["ptx_pixel_ok"] = (
    gdf_joined["filt_land"]
    & gdf_joined["filt_protected"]
    & gdf_joined["filt_elevation"]
    & gdf_joined["filt_slope"]
    & gdf_joined["filt_cf"]
    & gdf_joined["filt_water"]
    & gdf_joined["filt_pop"]
    & gdf_joined["filt_country"]
)

# ══════════════════════════════════════════════════════════════════════════════
# 5. SPATIAL THINNING
# ══════════════════════════════════════════════════════════════════════════════
n_before = int(gdf_joined["ptx_pixel_ok"].sum())
gdf_ok   = gdf_joined[gdf_joined["ptx_pixel_ok"]].copy()

if USE_SPATIAL_THINNING:
    # Step A — global grid
    gdf_ok["_grid_bin"] = (
        (gdf_ok["lat"] // GRID_DEG).astype(int).astype(str)
        + "_" +
        (gdf_ok["lon"] // GRID_DEG).astype(int).astype(str)
    )
    gdf_grid = (
        gdf_ok
        .groupby("_grid_bin", group_keys=False)
        .apply(lambda x: x.sample(1, random_state=42))
        .drop(columns="_grid_bin")
        .reset_index(drop=True)
    )
    gdf_ok.drop(columns="_grid_bin", inplace=True)

    # Step B — per-country cap
    if "NAME_EN" in gdf_grid.columns:
        gdf_thinned = (
            gdf_grid
            .groupby("NAME_EN", group_keys=False)
            .apply(lambda x: x.sample(n=min(len(x), MAX_PER_COUNTRY), random_state=42))
            .reset_index(drop=True)
        )
    else:
        gdf_thinned = gdf_grid.copy()

    # Step C — per-country minimum guarantee
    if "NAME_EN" in gdf_thinned.columns:
        represented = set(gdf_thinned["NAME_EN"].dropna())
        eligible     = set(gdf_ok["NAME_EN"].dropna())
        missing      = eligible - represented
        if missing:
            extras = (
                gdf_ok[gdf_ok["NAME_EN"].isin(missing)]
                .groupby("NAME_EN", group_keys=False)
                .apply(lambda x: x.sample(1, random_state=42))
            )
            gdf_thinned = pd.concat([gdf_thinned, extras], ignore_index=True)
else:
    gdf_thinned = gdf_ok.copy()

n_after = len(gdf_thinned)

# ══════════════════════════════════════════════════════════════════════════════
# 6. SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
n = len(gdf_joined)
print(f"\nTotal pixels: {n:,}")
print(f"  On land:                                  {int(gdf_joined['on_land'].sum()):,}")
print(f"  Excluded by land class (agri+urban+...):  {int((~gdf_joined['filt_land']).sum()):,}")
print(f"  Excluded by protected area:               {int((~gdf_joined['filt_protected']).sum()):,}")
print(f"  Excluded by elevation:                    {int((~gdf_joined['filt_elevation']).sum()):,}")
print(f"  Excluded by slope (>{MAX_SLOPE_DEG}deg):               {int((~gdf_joined['filt_slope']).sum()):,}")

if USE_CF_FILTER:
    print(f"  Excluded by min. capacity factor:         {int((~gdf_joined['filt_cf']).sum()):,}")
else:
    print("  Excluded by min. capacity factor:         DISABLED")

if USE_WATER_FILTER:
    print(f"  Excluded by water (arid + far coast):     {int((~gdf_joined['filt_water']).sum()):,}")
else:
    print("  Excluded by water (arid + far coast):     DISABLED")

print(f"  Excluded by pop density (<{MIN_POP_DENSITY} p/km2):   {int((~gdf_joined['filt_pop']).sum()):,}")
print(f"  Excluded by unclaimed territory:          {int((~gdf_joined['filt_country']).sum()):,}")
print(f"  -- Passing ALL filters:                   {n_before:,}  ({100*n_before/n:.1f}%)")

if USE_SPATIAL_THINNING:
    print(f"  -- After {GRID_DEG}deg grid + country bounds:      {n_after:,}")
else:
    print(f"  -- Spatial thinning disabled:             {n_after:,}")

print(f"\nLand class distribution (selected):\n{gdf_thinned['land_class'].value_counts(dropna=False).to_string()}")

if "NAME_EN" in gdf_thinned.columns:
    print(f"\nTop 10 countries by pixel count:\n{gdf_thinned['NAME_EN'].value_counts().head(10).to_string()}")


In [ ]:
# Plot pixels passing PtX filter
if "ptx_pixel_ok" not in gdf_joined.columns:
    raise KeyError("Column 'ptx_pixel_ok' not found. Run the filter cell first.")

plot_df = gdf_joined.copy()
plot_df["ptx_pixel_ok"] = plot_df["ptx_pixel_ok"].fillna(False).astype(bool)
# Use thinned pixels if available, otherwise fall back to all passing pixels
passed = gdf_thinned if "gdf_thinned" in globals() else plot_df[plot_df["ptx_pixel_ok"]]

fig = plt.figure(figsize=(10, 5), dpi=200)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#f7f7f7", linewidth=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
ax.add_feature(cfeature.BORDERS, linewidth=0.2)
ax.set_extent([-140, 170, -58, 70], crs=ccrs.PlateCarree())

# Background: all sampled pixels
ax.scatter(
    plot_df["lon"], plot_df["lat"],
    s=3, c="lightgray", alpha=0.35,
    transform=ccrs.PlateCarree(), label="All pixels", zorder=2
)

# Foreground: PtX-eligible pixels
ax.scatter(
    passed["lon"], passed["lat"],
    s=7, c="#1b9e77", alpha=0.9,
    transform=ccrs.PlateCarree(), label="Pass filter (ptx_pixel_ok)", zorder=3
)

share = 100 * len(passed) / len(plot_df) if len(plot_df) else 0
ax.set_title(f"PtX eligible pixels: {len(passed):,} / {len(plot_df):,} ({share:.1f}%)", fontsize=10)
ax.legend(loc="lower left", frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# --- ADD COUNTRY METADATA TO DATASET ---
# Reshape gdf_joined data back to (lat, lon) grid
shape = (len(ds.lat), len(ds.lon))

# Add ISO_A2 as a data variable
if 'ISO_A2' in gdf_joined.columns:
    ds['ISO_A2'] = xr.DataArray(
        gdf_joined['ISO_A2'].values.reshape(shape),
        coords={'lat': ds.lat, 'lon': ds.lon},
        dims=('lat', 'lon')
    )

# Add NAME_EN as a data variable
if 'NAME_EN' in gdf_joined.columns:
    ds['NAME_EN'] = xr.DataArray(
        gdf_joined['NAME_EN'].values.reshape(shape),
        coords={'lat': ds.lat, 'lon': ds.lon},
        dims=('lat', 'lon')
    )

# Add CONTINENT as a data variable  
if 'CONTINENT' in gdf_joined.columns:
    ds['CONTINENT'] = xr.DataArray(
        gdf_joined['CONTINENT'].values.reshape(shape),
        coords={'lat': ds.lat, 'lon': ds.lon},
        dims=('lat', 'lon')
    )

print("Country metadata added to dataset")
print(f"Dataset now has: {list(ds.data_vars)}")

In [ ]:
# Step D — force LOCATIONS_SENS case study sites into the thinned set
from config import LOCATIONS_SENS
for _, _, lat_s, lon_s in LOCATIONS_SENS:
    nearest_idx = (
        (gdf_joined["lat"] - lat_s) ** 2 + (gdf_joined["lon"] - lon_s) ** 2
    ).idxmin()
    if nearest_idx not in gdf_thinned.index:
        gdf_thinned = pd.concat(
            [gdf_thinned, gdf_joined.loc[[nearest_idx]]],
            ignore_index=False
        )
        print(f"  Added case study site ({lat_s}, {lon_s}) → row {nearest_idx}")

gdf_thinned = gdf_thinned[~gdf_thinned.index.duplicated()]
n_after = len(gdf_thinned)

# Finally prepare xarray by adding solar and wind hourly capacity factor:

In [ ]:
# Example: create your time dimension
TIME = pd.date_range("2025-01-01", periods=8760, freq="h")  # one year hourly

# Assuming your dataset already has lat/lon
lat = ds.lat.values
lon = ds.lon.values

# Add the time coordinate if not already present
ds = ds.assign_coords(time=TIME)

# --- initialize dynamic hourly variables ---
dynamic_cols = ["cf_solar", "cf_wind"]

for col in dynamic_cols:
    ds[col] = xr.DataArray(
        np.full((len(TIME), len(lat), len(lon)), np.nan, dtype=float),
        coords={"time": TIME, "lat": lat, "lon": lon},
        dims=("time", "lat", "lon"),
    )
ds

In [ ]:
# --- EXPLICITLY REBUILD AND STORE PTX MASKS BEFORE SAVING ---

shape = (len(ds.lat), len(ds.lon))

if "ptx_pixel_ok" not in gdf_joined.columns:
    raise KeyError("gdf_joined['ptx_pixel_ok'] is missing. Run the PtX filter cell first.")

if "gdf_thinned" not in globals():
    raise NameError("gdf_thinned is missing. Run the spatial thinning cell first.")

# Main passing mask
ptx_pixel_ok_flat = gdf_joined["ptx_pixel_ok"].fillna(False).astype(bool).to_numpy()

# Thinned mask from selected lat/lon pairs
ptx_thinned_ok_flat = np.zeros(len(gdf_joined), dtype=bool)
if len(gdf_thinned) > 0:
    thinned_keys = set(zip(gdf_thinned["lat"].to_numpy(), gdf_thinned["lon"].to_numpy()))
    joined_keys = list(zip(gdf_joined["lat"].to_numpy(), gdf_joined["lon"].to_numpy()))
    ptx_thinned_ok_flat = np.array([key in thinned_keys for key in joined_keys], dtype=bool)

# Force thinned mask to be a strict subset of ptx_pixel_ok
extra_mask = ptx_thinned_ok_flat & ~ptx_pixel_ok_flat
n_extra = int(extra_mask.sum())

if n_extra > 0:
    extra_points = gdf_joined.loc[extra_mask, ["lat", "lon", "NAME_EN", "ISO_A2"]].copy()
    print(f"Dropping {n_extra} thinned pixels that are not in ptx_pixel_ok:")
    print(extra_points.to_string(index=False))
    ptx_thinned_ok_flat[extra_mask] = False

ptx_pixel_ok_arr = ptx_pixel_ok_flat.reshape(shape)
ptx_thinned_ok_arr = ptx_thinned_ok_flat.reshape(shape)

# Write masks explicitly into ds
ds["ptx_pixel_ok"] = xr.DataArray(
    ptx_pixel_ok_arr,
    coords={"lat": ds.lat, "lon": ds.lon},
    dims=("lat", "lon"),
)

ds["ptx_thinned_ok"] = xr.DataArray(
    ptx_thinned_ok_arr,
    coords={"lat": ds.lat, "lon": ds.lon},
    dims=("lat", "lon"),
)

print("Stored fresh PtX masks into ds")
print(f"ptx_pixel_ok total:   {int(ds['ptx_pixel_ok'].sum())}")
print(f"ptx_thinned_ok total: {int(ds['ptx_thinned_ok'].sum())}")

tmp = gdf_joined.loc[gdf_joined["ptx_pixel_ok"], ["NAME_EN"]].copy()
tmp_th = gdf_joined.loc[ptx_thinned_ok_flat, ["NAME_EN"]].copy()

print("\nTop countries in ptx_pixel_ok:")
print(tmp["NAME_EN"].value_counts().head(15).to_string())

print("\nTop countries in ptx_thinned_ok:")
print(tmp_th["NAME_EN"].value_counts().head(15).to_string())


In [ ]:
# PtX MODIFICATION: apply scenario-aware land + protected-area masks
shape = (len(ds.lat), len(ds.lon))
land_mask = None
land_mask_offgrid = None
prot_mask = None

if "ptx_land_ok" in gdf_joined.columns:
    land_mask = xr.DataArray(
        gdf_joined["ptx_land_ok"].values.reshape(shape),
        coords={"lat": ds.lat, "lon": ds.lon},
        dims=("lat", "lon"),
    )

if "ptx_land_ok_offgrid" in gdf_joined.columns:
    land_mask_offgrid = xr.DataArray(
        gdf_joined["ptx_land_ok_offgrid"].values.reshape(shape),
        coords={"lat": ds.lat, "lon": ds.lon},
        dims=("lat", "lon"),
    )

if "ptx_protected_ok" in gdf_joined.columns:
    prot_mask = xr.DataArray(
        gdf_joined["ptx_protected_ok"].values.reshape(shape),
        coords={"lat": ds.lat, "lon": ds.lon},
        dims=("lat", "lon"),
    )

def _apply_mask(base_ds, lmask, pmask):
    if lmask is not None and pmask is not None:
        return base_ds.where(lmask & pmask)
    elif lmask is not None:
        return base_ds.where(lmask)
    elif pmask is not None:
        return base_ds.where(pmask)
    return base_ds.copy()

masked_ds = _apply_mask(ds, land_mask, prot_mask)
masked_ds_offgrid = _apply_mask(ds, land_mask_offgrid, prot_mask)

print(f"Grid-connected valid pixels: {int(land_mask.sum()) if land_mask is not None else 'N/A'}")
print(f"Off-grid valid pixels:       {int(land_mask_offgrid.sum()) if land_mask_offgrid is not None else 'N/A'}")


In [ ]:
with open(OUTPUT_FILE_XARRAY_INIT, "wb") as f:
    pickle.dump(masked_ds, f)

print(f"Dataset saved to {OUTPUT_FILE_XARRAY_INIT}")
print(f"Active mask (ptx_thinned_ok): {int(masked_ds['ptx_thinned_ok'].sum()):,} pixels")
print(f"Active mask (ptx_pixel_ok):   {int(masked_ds['ptx_pixel_ok'].sum()):,} pixels")


# Important, EXPORT FILE:

In [ ]:
with open(OUTPUT_FILE_XARRAY_INIT, "wb") as f:
    pickle.dump(masked_ds, f)
print(f"Dataset saved to {OUTPUT_FILE_XARRAY_INIT}")
print(f"Active mask (ptx_thinned_ok): {int(masked_ds['ptx_thinned_ok'].sum()):,} pixels")